# Spark 4.1: Declarative Data Pipeline Workshop

## Using Spark Declarative Pipelines (SDP)

---

**Version:** Apache Spark 4.1.x  
**Approach:** Declarative (Spark Declarative Pipelines API)  
**Companion Notebook:** `spark40_imperative_pipeline.ipynb` (run side-by-side)

---

### Workshop Overview

This notebook teaches you how to build a **production-grade data pipeline** using the new **Spark Declarative Pipelines (SDP)** framework introduced in Apache Spark 4.1. You will:

1. Understand the business domain and data model (same as 4.0 notebook)
2. Build a medallion architecture using SDP decorators
3. Learn the declarative approach to defining transformations
4. Compare this approach with the imperative style in Spark 4.0

### How to Use This Workshop

**If running side-by-side with Spark 4.0 notebook:**
- Open both notebooks in JupyterLab (drag one tab to create split view)
- Execute cells in the same order in both notebooks
- Compare the code structure, verbosity, and approach
- Note the section numbers match between notebooks

### What is Spark Declarative Pipelines?

SDP is a framework donated by Databricks to Apache Spark that lets you **define** your pipeline transformations using decorators, rather than **executing** them imperatively. The framework handles:

- Automatic dependency resolution
- Execution ordering
- Checkpointing for streaming
- Validation without running (dry-run)

### Prerequisites

```bash
# Start the lakehouse stack
./lakehouse start all

# Generate and load test data
./lakehouse testdata generate --days 90
./lakehouse testdata load
```

---

## Section 1: The Business Domain

### What is a Ghost Kitchen?

A **ghost kitchen** (also called a virtual kitchen, cloud kitchen, or dark kitchen) is a professional food preparation facility that produces food exclusively for delivery. Unlike traditional restaurants, ghost kitchens have:

- **No dine-in customers** - delivery only
- **Multiple virtual brands** - one physical kitchen operates 5-10 different "restaurants"
- **Optimized for throughput** - designed for high-volume delivery orders
- **Data-driven operations** - every aspect is tracked and optimized

### Our Fictional Company: Casper's Kitchens

In this workshop, we're building a data pipeline for **Casper's Kitchens**, a ghost kitchen operator with:

| Metric | Value |
|--------|-------|
| Virtual Brands | 20 (Burger Republic, Wok This Way, Pizza Planet, etc.) |
| Markets | 4 cities (San Francisco, Silicon Valley, Seattle, Austin) |
| Menu Items | 160 items across 10 food categories |
| Daily Orders | ~835 per day (varies by hour and day of week) |
| Data Volume | ~14.5M order lifecycle events over 90 days |

### The Business Questions We're Solving

The data team at Casper's Kitchens needs to answer:

1. **Operations:** What are the average kitchen prep times by brand? Where are bottlenecks?
2. **Delivery:** What's the average delivery time by location? What's the P95?
3. **Revenue:** Which brands generate the most revenue? What's the average order value?
4. **Growth:** Which brands are growing vs declining? How does demand vary by hour?

### Why a Data Pipeline?

Raw operational data is **messy and hard to query**. We need to:

1. **Clean** - Remove nulls, fix data quality issues
2. **Enrich** - Parse nested JSON, add computed fields
3. **Aggregate** - Pre-compute metrics for fast dashboards
4. **Organize** - Structure data in layers for different use cases

---

## Section 2: The Data Model

### Order Lifecycle Events

Every order generates multiple events as it moves through the fulfillment process:

```
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│ order_created   │────▶│ kitchen_started │────▶│ kitchen_finished│
└─────────────────┘     └─────────────────┘     └─────────────────┘
                                                        │
                                                        ▼
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│    delivered    │◀────│driver_picked_up │◀────│   order_ready   │
└─────────────────┘     └─────────────────┘     └─────────────────┘
        ▲                       │
        │               ┌───────┴───────┐
        └───────────────│  driver_ping  │ (every 60 seconds)
                        └───────────────┘
```

### Event Schema

Each event contains:

| Field | Type | Description |
|-------|------|-------------|
| `event_id` | string | Unique identifier for this event |
| `event_type` | string | One of: order_created, kitchen_started, etc. |
| `ts` | string | ISO timestamp of the event |
| `order_id` | string | Links all events for one order |
| `location_id` | int | Which city/kitchen processed this |
| `sequence` | int | Order of events (1, 2, 3...) |
| `body` | string | JSON with event-specific data |

### The `body` JSON Field

The `body` field contains different data depending on event type:

```json
// order_created event
{
  "brand_id": 5,
  "item_ids": "[12, 34, 56]",
  "total": 45.97,
  "lat": 37.7749,
  "lng": -122.4194
}

// driver_ping event
{
  "driver_id": "DRV-1234",
  "lat": 37.7812,
  "lng": -122.4098
}
```

### Dimension Tables

| Table | Records | Description |
|-------|---------|-------------|
| `dim_brands` | 20 | Virtual restaurant brands |
| `dim_categories` | 10 | Food categories (Pizza, Asian, etc.) |
| `dim_items` | 160 | Menu items with prices |
| `dim_locations` | 4 | Delivery markets/cities |

---

## Section 3: The Medallion Architecture

We'll organize our data using the **medallion architecture**, a data design pattern that organizes data into layers:

```
┌─────────────────────────────────────────────────────────────────────┐
│                         MEDALLION ARCHITECTURE                       │
├─────────────────────────────────────────────────────────────────────┤
│                                                                      │
│   ┌──────────┐      ┌──────────┐      ┌──────────┐                 │
│   │  BRONZE  │─────▶│  SILVER  │─────▶│   GOLD   │                 │
│   │  (Raw)   │      │ (Cleaned)│      │(Business)│                 │
│   └──────────┘      └──────────┘      └──────────┘                 │
│                                                                      │
│   • Raw ingestion    • Quality filters  • Pre-aggregated            │
│   • Schema-on-read   • Parsed JSON      • Business metrics          │
│   • Full fidelity    • Enriched fields  • Dashboard-ready           │
│                      • Joined dims                                   │
│                                                                      │
└─────────────────────────────────────────────────────────────────────┘
```

### Why Medallion?

| Layer | Purpose | Users |
|-------|---------|-------|
| **Bronze** | Preserve raw data exactly as received | Data engineers, debugging |
| **Silver** | Single source of truth, cleaned and enriched | Data scientists, analysts |
| **Gold** | Pre-computed metrics for specific use cases | Dashboards, executives |

### Our Tables

| Layer | Table | Description |
|-------|-------|-------------|
| Bronze | `bronze.dim_*` | Dimension tables (brands, items, locations, categories) |
| Bronze | `bronze.orders` | Raw order lifecycle events |
| Silver | `silver.orders_enriched` | Cleaned events with parsed JSON and time features |
| Silver | `silver.order_lifecycle` | One row per order with all timestamps and durations |
| Gold | `gold.hourly_metrics` | Orders and revenue by hour and location |
| Gold | `gold.delivery_performance` | Delivery time stats by date and location |
| Gold | `gold.brand_summary` | Brand-level performance metrics |

---

## Section 4: Environment Setup

### The Declarative Difference

In the **imperative approach** (Spark 4.0), you:
1. Create a SparkSession
2. Define functions that read, transform, AND write
3. Call functions in the correct order

In the **declarative approach** (SDP), you:
1. Define functions with decorators that ONLY return DataFrames
2. The framework handles SparkSession, execution order, and writes
3. Run via CLI: `spark-pipelines run`

### Notebook Adaptation

Since SDP is designed for CLI execution, we'll create a **mini pipeline runner** that mimics the framework's behavior. This lets us:
- Actually use decorators (not just show them in comments)
- Demonstrate automatic dependency resolution
- Execute interactively while learning the patterns

In [ ]:
# =============================================================================
# SECTION 4: ENVIRONMENT SETUP
# =============================================================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
)
from functools import wraps
from typing import Callable, Dict, List, Set
import re

# Create SparkSession (in production SDP, this is done by the framework)
spark = SparkSession.builder \
    .appName("Spark41_Declarative_Pipeline") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print(f"Spark Version: {spark.version}")
print(f"Application Name: {spark.sparkContext.appName}")

In [ ]:
# =============================================================================
# MINI PIPELINE RUNNER - Mimics SDP Framework Behavior
# =============================================================================
#
# This class demonstrates what the SDP framework does under the hood:
#   1. Register decorated functions
#   2. Infer dependencies from spark.table() calls in function source
#   3. Build a DAG and execute in topological order
#   4. Handle writes automatically
#
# In production, you'd use: from pyspark import pipelines as dp
# =============================================================================

class Pipeline:
    """Mini SDP framework that mimics @dp.materialized_view behavior.
    
    This demonstrates the KEY DIFFERENCE from imperative:
    - You DEFINE tables with decorators
    - You DON'T call functions or write data yourself
    - The framework handles execution order and persistence
    """
    
    def __init__(self, name: str, catalog: str = "iceberg"):
        self.name = name
        self.catalog = catalog
        self.tables: Dict[str, dict] = {}  # name -> {func, layer, deps}
    
    def materialized_view(self, name: str, layer: str = "bronze"):
        """Decorator that registers a table definition.
        
        COMPARE WITH IMPERATIVE:
        - Imperative: def func(): ...; df.write.saveAsTable(); func()
        - Declarative: @pipeline.materialized_view(...) def func(): return df
        
        The decorator captures the function but DOESN'T execute it.
        Execution happens when pipeline.run() is called.
        """
        def decorator(func: Callable):
            # Infer dependencies from spark.table() calls in source
            import inspect
            source = inspect.getsource(func)
            deps = set(re.findall(r'spark\.table\(["\']([^"\']+)["\']\)', source))
            
            self.tables[name] = {
                'func': func,
                'layer': layer,
                'deps': deps,
            }
            
            @wraps(func)
            def wrapper():
                return func()
            return wrapper
        return decorator
    
    def _get_execution_order(self) -> List[str]:
        """Topologically sort tables based on dependencies.
        
        This is the MAGIC of declarative pipelines:
        - You define tables in any order
        - The framework figures out the correct execution order
        """
        visited: Set[str] = set()
        order: List[str] = []
        
        def visit(table_name: str):
            if table_name in visited:
                return
            visited.add(table_name)
            
            if table_name in self.tables:
                for dep in self.tables[table_name]['deps']:
                    # Convert full table name to short name
                    short_dep = dep.replace(f"{self.catalog}.", "")
                    visit(short_dep)
            order.append(table_name)
        
        for table_name in self.tables:
            visit(table_name)
        
        return order
    
    def run(self, layer: str = None) -> Dict[str, int]:
        """Execute the pipeline.
        
        COMPARE WITH IMPERATIVE:
        - Imperative: You call each function in order manually
        - Declarative: Call pipeline.run() and everything executes automatically
        """
        results = {}
        order = self._get_execution_order()
        
        print(f"\nExecuting pipeline: {self.name}")
        print(f"Execution order (auto-resolved): {order}")
        print("=" * 60)
        
        for table_name in order:
            if table_name not in self.tables:
                continue
            
            info = self.tables[table_name]
            if layer and info['layer'] != layer:
                continue
            
            full_name = f"{self.catalog}.{table_name}"
            deps = info['deps']
            
            print(f"\n[{info['layer'].upper()}] {table_name}")
            if deps:
                print(f"  Dependencies: {deps}")
            
            # Execute the function and get the DataFrame
            df = info['func']()
            
            # Framework handles the write (you don't do this in your code!)
            df.write.mode("overwrite").saveAsTable(full_name)
            
            # Query table for count (avoids re-scanning large DataFrames)
            count = spark.sql(f"SELECT COUNT(*) as cnt FROM {full_name}").collect()[0]['cnt']
            results[table_name] = count
            print(f"  → {full_name}: {count:,} rows")
        
        print("\n" + "=" * 60)
        print(f"Pipeline complete: {len(results)} tables created")
        return results

# Create our pipeline instance
# In production: from pyspark import pipelines as dp
pipeline = Pipeline("lakehouse_pipeline", catalog="iceberg")

print("\n✓ Pipeline framework ready")
print("\nNote: In production, you would use:")
print("  from pyspark import pipelines as dp")
print("  @dp.materialized_view(name='bronze.orders')")

---

## Section 5: Bronze Layer - Raw Data Ingestion

The Bronze layer is our **landing zone**. We ingest data with minimal transformation.

### The Declarative Pattern

Notice the difference from the imperative notebook:

| Imperative (4.0) | Declarative (4.1) |
|------------------|-------------------|
| `def load_brands(): df = ...; df.write.save(); return count` | `@pipeline.materialized_view(...) def dim_brands(): return df` |
| You call the function | Framework calls the function |
| You write to table | Framework writes to table |
| You track counts | Framework tracks counts |

In [ ]:
# =============================================================================
# SECTION 5.1: DEFINE DIMENSION TABLES (DECLARATIVE STYLE)
# =============================================================================
#
# KEY INSIGHT: We're using ACTUAL decorators here, not comments!
#
# Notice:
#   1. Each function is decorated with @pipeline.materialized_view
#   2. Functions ONLY return a DataFrame - no write statements
#   3. We don't call these functions - the pipeline runner does
# =============================================================================

@pipeline.materialized_view(name="bronze.dim_categories", layer="bronze")
def dim_categories():
    """Food categories dimension table.
    
    DECLARATIVE: Just return the DataFrame.
    The decorator handles table naming and persistence.
    """
    return spark.read.parquet("/data/dimensions/categories.parquet")


@pipeline.materialized_view(name="bronze.dim_brands", layer="bronze")
def dim_brands():
    """Ghost kitchen brands dimension table."""
    return spark.read.parquet("/data/dimensions/brands.parquet")


@pipeline.materialized_view(name="bronze.dim_items", layer="bronze")
def dim_items():
    """Menu items dimension table."""
    return spark.read.parquet("/data/dimensions/items.parquet")


@pipeline.materialized_view(name="bronze.dim_locations", layer="bronze")
def dim_locations():
    """Delivery locations dimension table."""
    return spark.read.parquet("/data/dimensions/locations.parquet")


print("✓ Dimension tables DEFINED (not executed yet!)")
print(f"  Registered tables: {list(pipeline.tables.keys())}")
print("\nNotice: We haven't written any data yet.")
print("The decorators just REGISTER the table definitions.")

In [ ]:
# =============================================================================
# SECTION 5.2: DEFINE ORDER EVENTS TABLE
# =============================================================================

@pipeline.materialized_view(name="bronze.orders", layer="bronze")
def orders_batch():
    """Order lifecycle events from batch parquet source.
    
    DECLARATIVE INSIGHT:
      This function is PURE - it has no side effects.
      It just describes WHAT the table should contain.
      
      The framework can:
        - Validate this without running (dry-run)
        - Execute in parallel with other independent tables
        - Retry on failure
    """
    df = spark.read.parquet("/data/events/orders_90d.parquet")
    return df.withColumn(
        "event_timestamp",
        f.to_timestamp(f.regexp_replace("ts", "T", " "))
    )


print("✓ Orders table DEFINED")
print(f"  All registered tables: {list(pipeline.tables.keys())}")

In [ ]:
# =============================================================================
# SECTION 5.3: RUN BRONZE LAYER
# =============================================================================
#
# NOW we execute - with a single call!
#
# COMPARE WITH IMPERATIVE:
#   - 4.0: Call each function manually in a loop
#   - 4.1: Call pipeline.run() and everything happens automatically
# =============================================================================

bronze_results = pipeline.run(layer="bronze")

In [ ]:
# =============================================================================
# SECTION 5.4: EXPLORE BRONZE DATA
# =============================================================================

print("BRANDS (Virtual Restaurants)")
print("=" * 50)
spark.table("iceberg.bronze.dim_brands").show(truncate=False)

In [ ]:
print("LOCATIONS (Delivery Markets)")
print("=" * 50)
spark.table("iceberg.bronze.dim_locations").show(truncate=False)

In [ ]:
print("ORDER EVENTS SCHEMA")
print("=" * 50)
spark.table("iceberg.bronze.orders").printSchema()

In [ ]:
print("EVENT TYPE DISTRIBUTION")
print("=" * 50)
spark.table("iceberg.bronze.orders") \
    .groupBy("event_type") \
    .count() \
    .orderBy(f.desc("count")) \
    .show()

### Section 5 Summary: Bronze Layer

**What we did differently from 4.0:**
- Used `@pipeline.materialized_view` decorators
- Functions ONLY return DataFrames - no write statements
- Single `pipeline.run()` call executed everything

**What the framework handled:**
- Table naming and catalog registration
- Write mode (overwrite)
- Row counting and reporting

---

## Section 6: Silver Layer - Data Transformation

The Silver layer is where we **clean, enrich, and transform** our data.

### Declarative Dependencies

In SDP, dependencies are **inferred automatically** from `spark.table()` calls:

```python
@pipeline.materialized_view(name="silver.orders_enriched", layer="silver")
def orders_enriched():
    orders = spark.table("iceberg.bronze.orders")       # DEPENDENCY!
    locations = spark.table("iceberg.bronze.dim_locations")  # DEPENDENCY!
    # ...
```

The framework parses the function source, finds `spark.table()` calls, and builds a DAG.

In [ ]:
# =============================================================================
# SECTION 6.1: DEFINE ORDERS_ENRICHED TABLE
# =============================================================================
#
# KEY INSIGHT: The spark.table() calls CREATE DEPENDENCIES.
# The framework will automatically ensure bronze tables run first.
# =============================================================================

@pipeline.materialized_view(name="silver.orders_enriched", layer="silver")
def orders_enriched():
    """Orders with parsed JSON body, time features, and location join.
    
    DEPENDENCIES (auto-inferred from spark.table calls):
      - iceberg.bronze.orders
      - iceberg.bronze.dim_locations
    
    The framework will run those tables BEFORE this one.
    """
    
    # These spark.table() calls CREATE DEPENDENCIES
    orders = spark.table("iceberg.bronze.orders")
    locations = spark.table("iceberg.bronze.dim_locations")
    
    # Filter nulls
    cleaned = orders.filter(
        f.col("event_id").isNotNull() &
        f.col("order_id").isNotNull() &
        f.col("event_timestamp").isNotNull()
    )
    
    # Parse JSON body
    body_schema = StructType([
        StructField("brand_id", IntegerType(), True),
        StructField("item_ids", StringType(), True),
        StructField("total", DoubleType(), True),
        StructField("lat", DoubleType(), True),
        StructField("lng", DoubleType(), True),
        StructField("driver_id", StringType(), True),
    ])
    
    enriched = cleaned.withColumn("body_parsed", f.from_json("body", body_schema))
    
    # Extract fields
    enriched = enriched.select(
        "event_id", "event_type", "event_timestamp", "ts", "ts_seconds",
        "order_id", "location_id", "sequence", "body",
        f.col("body_parsed.brand_id").alias("brand_id"),
        f.col("body_parsed.total").alias("order_total"),
        f.col("body_parsed.lat").alias("latitude"),
        f.col("body_parsed.lng").alias("longitude"),
        f.col("body_parsed.driver_id").alias("driver_id"),
    )
    
    # Add time features
    enriched = enriched.withColumns({
        "event_hour": f.hour("event_timestamp"),
        "event_day_of_week": f.dayofweek("event_timestamp"),
        "is_weekend": f.when(f.dayofweek("event_timestamp").isin(1, 7), True).otherwise(False),
        "event_date": f.to_date("event_timestamp"),
    })
    
    # Join with locations
    locations_lookup = locations.select(
        f.col("id").alias("location_id"),
        f.col("city").alias("city_name"),
    )
    
    enriched = enriched.join(f.broadcast(locations_lookup), on="location_id", how="left")
    
    # NO WRITE STATEMENT! Just return the DataFrame.
    return enriched


print("✓ orders_enriched DEFINED")
print(f"  Dependencies detected: {pipeline.tables['silver.orders_enriched']['deps']}")

In [ ]:
# =============================================================================
# SECTION 6.2: DEFINE ORDER_LIFECYCLE TABLE
# =============================================================================

@pipeline.materialized_view(name="silver.order_lifecycle", layer="silver")
def order_lifecycle():
    """Pivoted view with one row per completed order and duration metrics.
    
    DEPENDENCY: iceberg.silver.orders_enriched
      The framework ensures orders_enriched runs before this.
    """
    
    # This spark.table() call creates a dependency
    orders = spark.table("iceberg.silver.orders_enriched")
    
    # Pivot events to columns
    lifecycle = orders.groupBy("order_id", "location_id", "city_name").pivot(
        "event_type",
        ["order_created", "kitchen_started", "kitchen_finished", "order_ready",
         "driver_arrived", "driver_picked_up", "delivered"]
    ).agg(f.min("event_timestamp").alias("ts"))
    
    # Rename columns
    lifecycle = lifecycle.select(
        "order_id", "location_id", "city_name",
        f.col("order_created").alias("created_at"),
        f.col("kitchen_started").alias("kitchen_started_at"),
        f.col("kitchen_finished").alias("kitchen_finished_at"),
        f.col("order_ready").alias("order_ready_at"),
        f.col("driver_arrived").alias("driver_arrived_at"),
        f.col("driver_picked_up").alias("pickup_at"),
        f.col("delivered").alias("delivered_at"),
    )
    
    # Calculate durations
    lifecycle = lifecycle.withColumns({
        "kitchen_duration_min": (f.unix_timestamp("kitchen_finished_at") - f.unix_timestamp("kitchen_started_at")) / 60,
        "delivery_duration_min": (f.unix_timestamp("delivered_at") - f.unix_timestamp("pickup_at")) / 60,
        "total_duration_min": (f.unix_timestamp("delivered_at") - f.unix_timestamp("created_at")) / 60,
    })
    
    # Filter to completed orders and return
    return lifecycle.filter(f.col("delivered_at").isNotNull())


print("✓ order_lifecycle DEFINED")
print(f"  Dependencies detected: {pipeline.tables['silver.order_lifecycle']['deps']}")

In [ ]:
# =============================================================================
# SECTION 6.3: RUN SILVER LAYER
# =============================================================================
#
# The framework will:
#   1. See that orders_enriched depends on bronze tables (already done)
#   2. See that order_lifecycle depends on orders_enriched
#   3. Execute in correct order: orders_enriched → order_lifecycle
# =============================================================================

silver_results = pipeline.run(layer="silver")

In [ ]:
# =============================================================================
# SECTION 6.4: EXPLORE SILVER DATA
# =============================================================================

print("ENRICHED ORDERS SCHEMA")
print("=" * 50)
spark.table("iceberg.silver.orders_enriched").printSchema()

In [ ]:
print("SAMPLE ENRICHED ORDERS")
print("=" * 50)
spark.table("iceberg.silver.orders_enriched") \
    .select("event_type", "order_id", "city_name", "brand_id", "order_total", "event_hour", "is_weekend") \
    .filter(f.col("event_type") == "order_created") \
    .show(10)

In [ ]:
print("SAMPLE ORDER LIFECYCLE")
print("=" * 50)
spark.table("iceberg.silver.order_lifecycle") \
    .select(
        "order_id", "city_name",
        f.round("kitchen_duration_min", 1).alias("kitchen_min"),
        f.round("delivery_duration_min", 1).alias("delivery_min"),
        f.round("total_duration_min", 1).alias("total_min")
    ).show(10)

### Section 6 Summary: Silver Layer

**What we did differently from 4.0:**
- Dependencies declared implicitly via `spark.table()` calls
- Framework inferred the correct execution order
- No manual ordering of function calls needed

**What the framework handled:**
- Detected that `order_lifecycle` depends on `orders_enriched`
- Executed `orders_enriched` first, then `order_lifecycle`
- No print statements needed - framework reported progress

---

## Section 7: Gold Layer - Business Aggregations

The Gold layer contains **pre-aggregated metrics** for dashboards and reports.

In [ ]:
# =============================================================================
# SECTION 7.1: DEFINE HOURLY_METRICS TABLE
# =============================================================================

@pipeline.materialized_view(name="gold.hourly_metrics", layer="gold")
def hourly_metrics():
    """Hourly order metrics by location.
    
    DEPENDENCY: iceberg.silver.orders_enriched
    """
    orders = spark.table("iceberg.silver.orders_enriched")
    
    return orders.filter(f.col("event_type") == "order_created").groupBy(
        "event_date", "event_hour", "location_id", "city_name"
    ).agg(
        f.count("order_id").alias("order_count"),
        f.sum("order_total").alias("total_revenue"),
        f.avg("order_total").alias("avg_order_value"),
        f.countDistinct("brand_id").alias("unique_brands"),
    )

print("✓ hourly_metrics DEFINED")

In [ ]:
# =============================================================================
# SECTION 7.2: DEFINE DELIVERY_PERFORMANCE TABLE
# =============================================================================

@pipeline.materialized_view(name="gold.delivery_performance", layer="gold")
def delivery_performance():
    """Delivery performance metrics by date and location.
    
    DEPENDENCY: iceberg.silver.order_lifecycle
    """
    lifecycle = spark.table("iceberg.silver.order_lifecycle")
    
    return lifecycle.groupBy(
        f.to_date("created_at").alias("order_date"),
        "location_id", "city_name"
    ).agg(
        f.count("order_id").alias("completed_orders"),
        f.avg("kitchen_duration_min").alias("avg_kitchen_time_min"),
        f.avg("delivery_duration_min").alias("avg_delivery_time_min"),
        f.avg("total_duration_min").alias("avg_total_time_min"),
        f.percentile_approx("total_duration_min", 0.5).alias("median_total_time_min"),
        f.percentile_approx("total_duration_min", 0.95).alias("p95_total_time_min"),
    )

print("✓ delivery_performance DEFINED")

In [ ]:
# =============================================================================
# SECTION 7.3: DEFINE BRAND_SUMMARY TABLE
# =============================================================================

@pipeline.materialized_view(name="gold.brand_summary", layer="gold")
def brand_summary():
    """Brand-level summary metrics.
    
    DEPENDENCIES:
      - iceberg.silver.orders_enriched
      - iceberg.bronze.dim_brands
    """
    orders = spark.table("iceberg.silver.orders_enriched")
    brands = spark.table("iceberg.bronze.dim_brands")
    
    brand_metrics = orders.filter(f.col("event_type") == "order_created").groupBy("brand_id").agg(
        f.count("order_id").alias("total_orders"),
        f.sum("order_total").alias("total_revenue"),
        f.avg("order_total").alias("avg_order_value"),
        f.countDistinct("location_id").alias("locations_served"),
        f.min("event_date").alias("first_order_date"),
        f.max("event_date").alias("last_order_date"),
    )
    
    return brand_metrics.join(
        brands.select(f.col("id").alias("brand_id"), "name"),
        on="brand_id", how="left"
    ).select(
        "brand_id", f.col("name").alias("brand_name"),
        "total_orders", "total_revenue", "avg_order_value",
        "locations_served", "first_order_date", "last_order_date",
    )

print("✓ brand_summary DEFINED")
print(f"\nAll registered tables: {list(pipeline.tables.keys())}")

In [ ]:
# =============================================================================
# SECTION 7.4: RUN GOLD LAYER
# =============================================================================

gold_results = pipeline.run(layer="gold")

In [ ]:
# =============================================================================
# SECTION 7.5: EXPLORE GOLD DATA
# =============================================================================

print("TOP PERFORMING BRANDS")
print("=" * 50)
spark.table("iceberg.gold.brand_summary") \
    .select("brand_name", "total_orders", f.round("total_revenue", 2).alias("total_revenue"), f.round("avg_order_value", 2).alias("avg_order_value")) \
    .orderBy(f.desc("total_revenue")) \
    .show(10)

In [ ]:
print("DELIVERY PERFORMANCE BY CITY")
print("=" * 50)
spark.table("iceberg.gold.delivery_performance") \
    .groupBy("city_name") \
    .agg(
        f.sum("completed_orders").alias("total_orders"),
        f.round(f.avg("avg_total_time_min"), 1).alias("avg_time_min"),
        f.round(f.avg("p95_total_time_min"), 1).alias("p95_time_min")
    ) \
    .orderBy(f.desc("total_orders")) \
    .show()

---

## Section 8: Pipeline Summary

### Dependency Graph (Auto-Resolved by Framework)

```
                    ┌────────────────┐
                    │  dim_locations │
                    └───────┬────────┘
                            │
┌─────────┐         ┌───────▼────────┐         ┌────────────────┐
│ orders  │────────▶│orders_enriched │────────▶│ hourly_metrics │
└─────────┘         └───────┬────────┘         └────────────────┘
                            │                          │
                            │                  ┌───────▼────────┐
                            │                  │ brand_summary  │◀── dim_brands
                            │                  └────────────────┘
                    ┌───────▼────────┐
                    │order_lifecycle │
                    └───────┬────────┘
                            │
                    ┌───────▼────────┐
                    │delivery_perf   │
                    └────────────────┘
```

**In SDP, this graph is built automatically from `spark.table()` calls.**

---

## Section 9: Declarative Approach - Tradeoffs

### Advantages

| Advantage | Why It Matters |
|-----------|----------------|
| **Automatic dependency resolution** | No manual function call ordering |
| **Less boilerplate** | No write statements in your code |
| **Built-in validation** | `dry-run` catches errors before execution |
| **Automatic parallelism** | Independent tables run concurrently |
| **CLI integration** | `spark-pipelines run` for production |

### Disadvantages

| Disadvantage | Impact |
|--------------|--------|
| **Framework dependency** | Requires Spark 4.1+ with SDP |
| **Less control** | Framework decides execution details |
| **Debugging harder** | Errors surface in framework code |
| **Learning curve** | New decorator/DAG concepts |

### Side-by-Side Comparison

| Aspect | Imperative (4.0) | Declarative (4.1) |
|--------|------------------|-------------------|
| Function body | Read + Transform + Write | Read + Transform + Return |
| Execution | Manual function calls | `pipeline.run()` |
| Dependencies | Implicit (call order) | Explicit (`spark.table()`) |
| Progress tracking | Print statements | Framework provides |
| Production CLI | Custom scripts | `spark-pipelines run` |

---

## Section 10: Production Usage

In production, your pipeline file (`scripts/pipeline_sdp.py`) looks like:

```python
from pyspark import pipelines as dp
from pyspark.sql import functions as f

@dp.materialized_view(name="bronze.orders")
def orders_batch():
    df = spark.read.parquet("/data/events/orders_90d.parquet")
    return df.withColumn("event_timestamp", ...)

@dp.materialized_view(name="silver.orders_enriched")
def orders_enriched():
    orders = spark.table("iceberg.bronze.orders")
    locations = spark.table("iceberg.bronze.dim_locations")
    return enriched_df

# ... more tables ...
```

Run with:
```bash
spark-pipelines dry-run --spec scripts/spark-pipeline.yml  # Validate
spark-pipelines run --spec scripts/spark-pipeline.yml      # Execute
```

In [ ]:
print("\n" + "=" * 60)
print("WORKSHOP COMPLETE")
print("=" * 60)
print("\nYou've built a complete declarative data pipeline.")
print("\nKey takeaways:")
print("  1. @decorator registers tables, doesn't execute them")
print("  2. spark.table() calls create dependencies")
print("  3. pipeline.run() executes in correct order")
print("  4. No write statements in your functions")
print("\nCompare with spark40_imperative_pipeline.ipynb to see the difference!")